In [2]:
import torch
import typing as tt
import numpy as np
from sentence_transformers import SentenceTransformer

# Static word embeddings

In [3]:
model = SentenceTransformer(
    'sentence-transformers/average_word_embeddings_glove.6B.300d',
    device="cpu"
)
model

SentenceTransformer(
  (0): WordEmbeddings({'tokenizer_class': 'sentence_transformers.sentence_transformer.modules.tokenizer.whitespace.WhitespaceTokenizer', 'update_embeddings': False, 'max_seq_length': 1000000})
  (1): Pooling({'embedding_dimension': 300, 'pooling_mode': 'mean', 'include_prompt': True})
)

In [5]:
model.max_seq_length

1000000

In [6]:
print(model[0].emb_layer)
print(model[0].emb_layer.weight.shape)

Embedding(400001, 300)
torch.Size([400001, 300])


In [7]:
tokenizer = model[0].tokenizer
print(tokenizer)
print(len(tokenizer.get_vocab()))
print(tokenizer.get_vocab()[:20])

400001
['PADDING_TOKEN', 'the', ',', '.', 'of', 'to', 'and', 'in', 'a', '"', "'s", 'for', '-', 'that', 'on', 'is', 'was', 'said', 'with', 'he']


In [8]:
tokenizer.word2idx['unintentional']

33739

## Word similarity

In [9]:
tokenizer = model[0].tokenizer
vocab = tokenizer.get_vocab()
word = "zurich"

idx = tokenizer.word2idx[word]
print("index", idx)
idx_t = torch.LongTensor([idx])
emb = model[0].emb_layer(idx_t)
print("emb shape", emb.shape)
print(emb)

index 8549
emb shape torch.Size([1, 300])
tensor([[-0.1131, -0.0533, -0.3217,  0.6362,  0.3280, -0.3287,  0.2292, -0.5596,
         -0.4363, -0.4929,  0.4543, -0.4889, -0.0079, -0.3734,  0.1456, -0.1328,
          0.2800,  0.6924, -0.0828, -0.1738,  0.2720, -0.5756, -0.5459, -0.1834,
         -0.3191, -0.0956,  0.1038,  0.1025, -0.4443, -0.4250, -0.3745, -0.5149,
          0.2474,  0.7552, -0.4292, -0.5165,  0.1155,  0.4098, -0.1654, -0.3672,
         -0.7025, -0.6253, -0.5423,  0.2298,  0.3456,  0.7394, -0.2693,  0.9754,
         -0.1108,  0.3309, -0.6725, -0.3202, -0.1215,  0.2402,  0.3163,  0.1106,
         -0.2856, -0.1253,  0.5182, -0.1895, -0.7142,  0.2969, -0.2178, -0.0456,
         -0.2402,  0.4773, -0.3109, -0.3433, -0.4915, -0.8149,  0.2071, -0.1620,
          0.5094,  0.2165, -0.0178,  0.2304,  0.0236,  0.0053, -0.0291,  0.0380,
         -0.7728, -0.1087, -0.4988,  0.5972,  0.0842,  0.1070, -0.2578, -0.0897,
         -0.8782,  0.3065,  0.3895, -0.1384,  0.6636, -0.2586,  0.9

In [10]:
sim = model.similarity(emb, model[0].emb_layer.weight)
print("sim shape", sim.shape)
print("similarity", sim)
order = torch.argsort(sim, descending=True)
print("order", order)

sim shape torch.Size([1, 400001])
similarity tensor([[ 0.0000,  0.0937,  0.1452,  ..., -0.0685, -0.0371, -0.1092]])
order tensor([[  8549,  27320,  12949,  ...,  83905, 349119, 243152]])


In [11]:
for idx in order[0, :10]:
    print(int(idx), vocab[idx])

8549 zurich
27320 zürich
12949 basel
2312 switzerland
17408 bern
4716 munich
5334 frankfurt
1850 swiss
17788 lausanne
33877 lucerne


In [12]:
def show_similar(model: SentenceTransformer, word: str, top_k: int = 10):
    vocab = model[0].tokenizer.get_vocab()
    idx = model[0].tokenizer.word2idx[word]
    idx_t = torch.LongTensor([idx])
    emb = model[0].emb_layer(idx_t)
    sim = model.similarity(emb, model[0].emb_layer.weight)
    order = torch.argsort(sim, descending=True)
    for idx in order[0, :top_k]:
        print(f"{vocab[idx]}: {sim[0, idx]:.4f}")

In [13]:
show_similar(model, "amsterdam")

amsterdam: 1.0000
rotterdam: 0.6486
schiphol: 0.5740
utrecht: 0.5609
netherlands: 0.5472
frankfurt: 0.5457
brussels: 0.5311
antwerp: 0.5171
dutch: 0.5104
stockholm: 0.5074


In [14]:
show_similar(model, "collider")

collider: 1.0000
hadron: 0.7627
cern: 0.6691
lhc: 0.6441
tevatron: 0.5461
fermilab: 0.4959
accelerator: 0.4718
rhic: 0.4536
smasher: 0.4529
particle: 0.4354


In [15]:
show_similar(model, "bureaucracy")

bureaucracy: 1.0000
bureaucratic: 0.7578
bureaucracies: 0.6966
bloated: 0.6162
bureaucrats: 0.6065
inefficiency: 0.5700
inefficient: 0.5416
streamline: 0.5411
streamlining: 0.5318
cumbersome: 0.5316


In [16]:
show_similar(model, "apple")

apple: 1.0000
iphone: 0.5987
macintosh: 0.5836
ipod: 0.5761
microsoft: 0.5664
ipad: 0.5628
intel: 0.5458
ibm: 0.5286
google: 0.5282
imac: 0.5073


## Sentence similarity

In [17]:
emb = model.encode([
    "The company increased its profits",
    "The firm's earnings grew",
    "The company opened a new office",
])
emb.shape

(3, 300)

In [19]:
model.similarity(emb, emb)

tensor([[1.0000, 0.7485, 0.6086],
        [0.7485, 1.0000, 0.4014],
        [0.6086, 0.4014, 1.0000]])

In [20]:
# BUT
emb = model.encode([
    "The dog bites the man.",
    "The man bites the dog.",
])
model.similarity(emb, emb)

tensor([[1.0000, 1.0000],
        [1.0000, 1.0000]])

## Semantic arithmetics

In [21]:
def get_emb(model: SentenceTransformer, word: str) -> torch.Tensor:
    idx = model[0].tokenizer.word2idx[word]
    idx_t = torch.LongTensor([idx])
    return model[0].emb_layer(idx_t)

In [22]:
def show_similar(model: SentenceTransformer, vector: torch.Tensor):
    sim = model.similarity(w, model[0].emb_layer.weight)
    order = torch.argsort(sim, descending=True)
    for idx in order[0, :10]:
        print(f"{vocab[idx]}: {sim[0, idx]:.4f}")

In [23]:
w1 = get_emb(model, "paris")
w2 = get_emb(model, "france")
w3 = get_emb(model, "italy")
w = w1 - w2 + w3
show_similar(model, w)

rome: 0.7268
italy: 0.6836
milan: 0.6613
italian: 0.6134
turin: 0.5653
bologna: 0.5615
naples: 0.5558
venice: 0.5462
paris: 0.5435
florence: 0.4982


In [24]:
w1 = get_emb(model, "sitting")
w2 = get_emb(model, "sit")
w3 = get_emb(model, "walk")
w = w1 - w2 + w3
show_similar(model, w)

walking: 0.7905
walk: 0.7896
walked: 0.6548
walks: 0.6448
sitting: 0.5766
stairs: 0.4799
strolling: 0.4333
talking: 0.4266
standing: 0.4261
woman: 0.4218


In [25]:
w1 = get_emb(model, "obama")
w2 = get_emb(model, "usa")
w3 = get_emb(model, "france")
w = w1 - w2 + w3
show_similar(model, w)

obama: 0.6508
sarkozy: 0.6428
france: 0.6165
barack: 0.6118
chirac: 0.5712
french: 0.5228
bush: 0.5164
clinton: 0.5067
paris: 0.4698
rodham: 0.4661


In [26]:
w1 = get_emb(model, "burgundy")
w2 = get_emb(model, "wine")
w3 = get_emb(model, "germany")
w = w1 - w2 + w3
show_similar(model, w)

germany: 0.7131
saxony: 0.5253
austria: 0.5158
denmark: 0.4741
thuringia: 0.4739
netherlands: 0.4640
hungary: 0.4607
bavaria: 0.4570
german: 0.4556
poland: 0.4545


# Transformer embeddings

In [27]:
model = SentenceTransformer(
    'microsoft/deberta-v3-small',
    device="cpu"
)
model

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'DebertaV2Model'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
)

In [28]:
model.max_seq_length

512

In [29]:
emb = model.encode([
    "The company increased its profits",
    "The firm's earnings grew",
    "The company opened a new office",
])
model.similarity(emb, emb)

tensor([[1.0000, 0.9553, 0.9368],
        [0.9553, 1.0000, 0.9252],
        [0.9368, 0.9252, 1.0000]])

In [30]:
emb = model.encode([
    "The dog bites the man.",
    "The man bites the dog.",
])
model.similarity(emb, emb)

tensor([[1.0000, 0.9831],
        [0.9831, 1.0000]])

In [31]:
emb = model.encode([
    "A man is playing a guitar.",
    "Someone is performing music.",
    "A woman is cooking dinner.",
    "The guitarist bought an expensive instrument.",
])
model.similarity(emb, emb)

tensor([[1.0000, 0.9128, 0.9723, 0.9225],
        [0.9128, 1.0000, 0.9256, 0.9411],
        [0.9723, 0.9256, 1.0000, 0.9308],
        [0.9225, 0.9411, 0.9308, 1.0000]])

# Sentence embeddings

In [32]:
model = SentenceTransformer(
    'BAAI/bge-small-en-v1.5',
    device="cpu"
)
model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [33]:
model.max_seq_length

512

In [34]:
emb = model.encode([
    "The company increased its profits",
    "The firm's earnings grew",
    "The company opened a new office",
])
model.similarity(emb, emb)

tensor([[1.0000, 0.8409, 0.7379],
        [0.8409, 1.0000, 0.6877],
        [0.7379, 0.6877, 1.0000]])

In [35]:
emb = model.encode([
    "The dog bites the man.",
    "The man bites the dog.",
])
model.similarity(emb, emb)

tensor([[1.0000, 0.9802],
        [0.9802, 1.0000]])

In [36]:
emb = model.encode([
    "A man is playing a guitar.",
    "Someone is performing music.",
    "A woman is cooking dinner.",
    "The guitarist bought an expensive instrument.",
])
model.similarity(emb, emb)

tensor([[1.0000, 0.8134, 0.4625, 0.7241],
        [0.8134, 1.0000, 0.5634, 0.6829],
        [0.4625, 0.5634, 1.0000, 0.4168],
        [0.7241, 0.6829, 0.4168, 1.0000]])

# Max sequence length

In [77]:
text = "The guitarist bought an expensive instrument." *100
e1 = model.encode(text)
e2 = model.encode(text + "Dog barked")
max(e1-e2)

np.float32(0.0)

In [70]:
tokenizer = model[0].tokenizer
tokenizer

BertTokenizer(name_or_path='BAAI/bge-small-en-v1.5', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [81]:
tokenizer.__module__

'transformers.models.bert.tokenization_bert'

In [71]:
len(tokenizer.encode(text))

702

In [87]:
import data
texts, _ = data.load_dataset()
print(len(texts))

count_longer = 0
count_chunks = 0
max_chunks = 0

for t in texts:
    l = len(model[0].tokenizer.encode(t))
    chunks = (l // 512) + 1
    count_chunks += chunks
    max_chunks = max(max_chunks, chunks)
    if l > 512:
        count_longer += 1
count_longer, count_chunks, max_chunks

50000


(7214, 58378, 7)